In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "http://quotes.toscrape.com"
r = requests.get(url)

soup = BeautifulSoup(r.text, "html.parser")

quotes = [q.text for q in soup.find_all("span", class_="text")]

df = pd.DataFrame(quotes, columns=["quote"])
df.to_csv("raw_quotes.csv", index=False)

df.head()


,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [11]:
df = pd.read_csv("raw_quotes.csv")

df["quote"] = df["quote"].str.strip().str.lower()
df = df.drop_duplicates()

df.to_csv("clean_quotes.csv", index=False)

df.head()


,quote
0,“the world as we have created it is a process ...
1,"“it is our choices, harry, that show what we t..."
2,“there are only two ways to live your life. on...
3,"“the person, be it gentleman or lady, who has ..."
4,"“imperfection is beauty, madness is genius and..."


In [12]:
import sqlite3

conn = sqlite3.connect("quotes.db")

df.to_sql("quotes", conn, if_exists="replace", index=False)

print("Data inserted into database")


Data inserted into database


In [15]:
import schedule
import time

def job():
    print("Running workflow...")
    
    # scraping
    import requests
    from bs4 import BeautifulSoup
    import pandas as pd
    
    r = requests.get("http://quotes.toscrape.com")
    soup = BeautifulSoup(r.text, "html.parser")
    quotes = [q.text for q in soup.find_all("span", class_="text")]
    df = pd.DataFrame(quotes, columns=["quote"])
    
    # cleaning
    df["quote"] = df["quote"].str.strip().str.lower()
    df = df.drop_duplicates()
    
    # database
    import sqlite3
    conn = sqlite3.connect("quotes.db")
    df.to_sql("quotes", conn, if_exists="replace", index=False)
    
    print("Workflow complete\n")

schedule.every(10).seconds.do(job)

for _ in range(3):  # run 3 times only for demo
    schedule.run_pending()
    time.sleep(10)


Running workflow...
Workflow complete

Running workflow...
Workflow complete

Running workflow...
Workflow complete

Running workflow...
Workflow complete

Running workflow...
Workflow complete

Running workflow...
Workflow complete

Running workflow...
Workflow complete

Running workflow...
Workflow complete



Using schedule library, the entire pipeline runs automatically every 10 seconds — scraping, cleaning, and database insertion without manual work.